In [2]:
# from tensorflow.test import is_built_with_cuda
# is_built_with_cuda()

In [4]:
# !nvidia-smi

In [106]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [107]:
import numpy as np
import os
import zipfile

In [108]:
import random
import string

In [13]:
path_to_zip = tf.keras.utils.get_file(
    'spa-eng.zip',
    origin='http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip',
    extract=True
)
path_to_zip

'/home/impubuntu/.keras/datasets/spa-eng_extracted'

In [286]:
path_to_file = os.path.join(
    path_to_zip,
    'spa-eng',
    'spa.txt'
)
path_to_file

'/home/impubuntu/.keras/datasets/spa-eng_extracted/spa-eng/spa.txt'

In [287]:
# -- 파일에서 데이터 불러오기

In [288]:
# 구두점 제거
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [289]:
punc_trans = str.maketrans('', '', string.punctuation) # (변경전문자,변경후문자,제외시킬문자)

In [290]:
txt = "Hello! "
print(txt)
txt = txt.translate(punc_trans)
print(txt)

Hello! 
Hello 


In [291]:
def preprocess_sentence(sentence):
    sentence = sentence.translate(punc_trans) # 구두점 제거
    sentence = sentence.lower() # 소문자로 변환
    sentence = "<start> " + sentence + " <end>"
    return sentence

In [339]:
def create_dataset(path, nums):
    text_pairs = [] # [ENG, SPA]
    with open(path) as f:
        lines = f.read().strip().split("\n")
        print(f'Total {len(lines)} lines.')

        cnt = 0
        for line in lines:
            eng, spa = line.split("\t")

            eng = preprocess_sentence(eng)
            spa = preprocess_sentence(spa)
            
            text_pairs.append((eng, spa))

            cnt += 1
            if cnt >= nums:
                break
    return np.array(text_pairs)

In [342]:
text_pairs = create_dataset(path_to_file, 10000)
text_pairs.shape

Total 118964 lines.


(10000, 2)

In [343]:
# 데이터셋 랜덤으로 5개 확인해보기
for _ in range(5):
    print(random.choice(text_pairs))

['<start> you made it <end>' '<start> lo han conseguido ustedes <end>']
['<start> i wont lose <end>' '<start> ¡no voy a perder <end>']
['<start> its a classic <end>' '<start> ese es un clásico <end>']
['<start> i am a teacher <end>' '<start> soy maestro <end>']
['<start> arent you hot <end>' '<start> ¿vosotros no tenéis calor <end>']


In [344]:
# 영어용 토크나이저 만들기(각 단어들에 숫자 인덱스 붙임)
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(text_pairs[:,0])
eng_sequences = eng_tokenizer.texts_to_sequences(text_pairs[:,0])

In [345]:
# 에스파뇰 토크나이저
spa_tokenizer = Tokenizer(filters='')
spa_tokenizer.fit_on_texts(text_pairs[:,1])
spa_sequences = spa_tokenizer.texts_to_sequences(text_pairs[:,1])

In [346]:
print('<start>' in spa_tokenizer.word_index)

True


In [347]:
# spa_tokenizer.word_index
text_pairs[:10,1]

array(['<start> ve <end>', '<start> vete <end>', '<start> vaya <end>',
       '<start> váyase <end>', '<start> hola <end>',
       '<start> ¡corre <end>', '<start> corred <end>',
       '<start> ¿quién <end>', '<start> ¡fuego <end>',
       '<start> ¡incendio <end>'], dtype='<U53')

In [348]:
# 시퀀스 확인해보기
for i in range(1):
    print(eng_sequences[i])
    print(text_pairs[i,0])
    print('\n')
    
    print(spa_sequences[i])
    print(text_pairs[i,1])
    print('-' * 20)

[1, 17, 2]
<start> go <end>


[1, 77, 2]
<start> ve <end>
--------------------


In [349]:
# 단어 사전 크기
eng_vocab_size = len(eng_tokenizer.word_index) + 1
spa_vocab_size = len(spa_tokenizer.word_index) + 1

print(eng_vocab_size)
print(spa_vocab_size)

# word_index는 1부터 시작하는데
# padding 문자의 인덱스(0)를 포함시키기 위해서 1을 더해줌 

2365
4962


In [350]:
# Padding

In [351]:
# 입력 언어(eng) 시퀀스 패딩
max_eng_seq_len = max( len(seq) for seq in eng_sequences )
encoder_input_data = pad_sequences(
    eng_sequences,
    maxlen=max_eng_seq_len,
    padding='post'
)

In [352]:
# 출력 언어(spa) 시퀀스 패딩
max_spa_seq_len = max( len(seq) for seq in spa_sequences )
decoder_input_data = pad_sequences(
    spa_sequences,
    maxlen=max_spa_seq_len,
    padding='post'
)

In [353]:
# 디코더의 타겟 데이터 만들기
# <start> hola mundo <end> -> hola mundo <end> <pad>
decoder_target_data = np.zeros_like(decoder_input_data)
# decoder_target_data.shape
for i, seq in enumerate(spa_sequences):
    decoder_target_data[i, :len(seq)-1] = seq[1:]

In [354]:
print(f'최대 입력 길이: {max_eng_seq_len}')
print(f'최대 출력 길이: {max_spa_seq_len}')
print('-' * 20)
print(f'인코더 입력 데이터: {encoder_input_data.shape}')
print(f'디코더 입력 데이터: {decoder_input_data.shape}')
print(f'디코더 타겟 데이터: {decoder_target_data.shape}')

최대 입력 길이: 7
최대 출력 길이: 10
--------------------
인코더 입력 데이터: (10000, 7)
디코더 입력 데이터: (10000, 10)
디코더 타겟 데이터: (10000, 10)


In [355]:
# 모델 준비
embedding_dim = 256 # 임베딩 벡터의 차원 수
lstm_units = 512 # LSTM 유닛 수(LSTM 레이어의 출력 차원)

In [356]:
# 인코더 만들기
encoder_inputs = Input(
    shape=(max_eng_seq_len,),
    name='encoder_inputs'
)

# Embedding: 정수로된 시퀀스를 밀집 벡터 시퀀스로 변환
encoder_embedding = Embedding(
    eng_vocab_size,
    embedding_dim,
    name='encoder_embedding'
)(encoder_inputs)

# LSTM: 시퀀스 처리 후 context vector를 반환
# return_state=True: 마지막 타임스텝의 hidden state, cell state를 반환
encoder_lstm = LSTM(
    lstm_units,
    return_state=True,
    name='encoder_lstm'
)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# 인코더의 상태(state_h, state_c)는 디코더로 전달될 문맥 벡터
encoder_states = [state_h, state_c]

In [357]:
# 디코더 만들기
decoder_inputs = Input(
    shape=(max_spa_seq_len,),
    name='decoder_inputs'
)

# Embedding: 타겟 언어의 단어사전 사용
decoder_embedding_layer = Embedding(
    spa_vocab_size,
    embedding_dim,
    name='decoder_embedding'
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

# LSTM: 인코더의 상태값을 초기 상태로 설정
# return_sequences=True: 모든 타입스텝의 출력을 반환
# (각 타임스텝에서 단어 예측을 위해)
decoder_lstm = LSTM(
    lstm_units,
    return_sequences=True,
    return_state=True,
    name='decoder_lstm'
)

# 훈련시에는 디코더 상태 사용안함
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states 
)

# Dense: 각 타입스텝의 LSTM 출력에 대해 다음 단어를 예측(소프트맥스)
decoder_dense = Dense(
    spa_vocab_size,
    activation='softmax',
    name='decoder_dense'
)
decoder_outputs = decoder_dense(decoder_outputs)

In [358]:
# 훈련 모델
# 입력: 인코더 입력, 디코더 입력
# 출력: 디코더 출력
training_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name='training_model'
)

training_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)
training_model.summary()

Model: "training_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 7, 256)    │    605,440 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 10, 256)   │  1,270,272 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 512),     │  1,574,912 │ encoder_embeddin… │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 10, 512), │  1,574,912 │ decoder_embeddin… │
│                     │ (None, 512),      │            │ encoder_lstm[0][… │
│                     │ (None, 512)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 10, 4962)  │  2,545,506 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,571,042 (28.88 MB)

 Trainable params: 7,571,042 (28.88 MB)

 Non-trainable params: 0 (0.00 B)

In [359]:
# 훈련시키기
batch_size=64
epochs = 100 # 50~100회는 해야함

In [360]:
# decoder_target_data를 3차원으로 확장
# (sparse_categorical_crossentropy는 각 타임스텝에 대해 레이블이 필요)
# (num_samples, max_spa_seq_len) -> (num_samples, max_spa_seq_len, 1)
decoder_target_data_for_loss = np.expand_dims(decoder_target_data, -1)
decoder_target_data_for_loss.shape

(10000, 10, 1)

In [ ]:
# 훈련 시작
# GPU 메모리 부족 발생하면 batch_size를 줄이거나 lstm_units를 줄여봐야됨
# CPU 사용하는 방법도 있음(tf.config.set_visible_devices([], 'GPU'))
try:
    history = training_model.fit(
        [encoder_input_data, decoder_input_data], # 입력 데이터
        decoder_target_data_for_loss, # 타겟 데이터(정답용)
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.2
    )
except Exception as e:
    print(e)

In [362]:
# 추론(번역) 모델 만들기
# 추론은 한번에 한단어씩만 생성하기 때문에 훈련 모델과 다름

# 1. encoder model
# 입력 문장을 받아 문맥 벡터(상태) 반환
encoder_model = Model(
    encoder_inputs, 
    encoder_states,
    name='encoder_model'
)

# 2. decoder model
# 인코더 상태와 이전 스텝에서 예측된 단어를 입력받아 다음 단어 예측
decoder_state_input_h = Input(
    shape=(lstm_units,),
    name='decoder_state_input_h'
)
decoder_state_input_c = Input(
    shape=(lstm_units,),
    name='decoder_state_input_c'
)
decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

# 추론 시 디코더 입력은 매 스텝마다 단어 하나씩만 들어옴(길이 1)
# Embedding레이어는 훈련에 사용했던 것 그대로 사용함
decoder_inputs_single_step = Input(
    shape=(1,),
    name='decoder_inputs_single_step'
)
decoder_embedding_single_step = decoder_embedding_layer(decoder_inputs_single_step)

# LSTM 레이어도 훈련 시 사용했던것 그대로 사용
# 초기 상태는 인코더 상태 + 이전 스텝의 디코더 상태
decoder_outputs_single_step, state_h_single_step, state_c_single_step = decoder_lstm(
    decoder_embedding_single_step,
    initial_state=decoder_states_inputs
)
decoder_states_single_step = [
    state_h_single_step,
    state_c_single_step
]

# Dense 레이어도 훈련 시 사용했던 것 그대로 사용
decoder_outputs_single_step = decoder_dense(decoder_outputs_single_step)

# 디코더 모델 정의
# 입력: 이전 스텝에서 예측한 단어 1개, 이전 스텝의 은닉,셀 상태
# 출력: 다음 단어에 대한 확률 분포, 현재 스템의 은닉, 셀 상태
decoder_model = Model(
    [decoder_inputs_single_step] + decoder_states_inputs,
    [decoder_outputs_single_step] + decoder_states_single_step,
    name='decoder_model'
)

In [363]:
encoder_model.summary()

Model: "encoder_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_inputs (InputLayer)     │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_embedding (Embedding)   │ (None, 7, 256)         │       605,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_lstm (LSTM)             │ [(None, 512), (None,   │     1,574,912 │
│                                 │ 512), (None, 512)]     │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,180,352 (8.32 MB)

 Trainable params: 2,180,352 (8.32 MB)

 Non-trainable params: 0 (0.00 B)

In [364]:
decoder_model.summary()

Model: "decoder_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_inputs_sin… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 1, 256)    │  1,270,272 │ decoder_inputs_s… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_inpu… │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_inpu… │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 512),  │  1,574,912 │ decoder_embeddin… │
│                     │ (None, 512),      │            │ decoder_state_in… │
│                     │ (None, 512)]      │            │ decoder_state_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 1, 4962)   │  2,545,506 │ decoder_lstm[1][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,390,690 (20.56 MB)

 Trainable params: 5,390,690 (20.56 MB)

 Non-trainable params: 0 (0.00 B)

In [365]:
# 단어 인덱스를 단어로 변환하기 위한 사전
rev_eng_char_index = { i: char for char, i in eng_tokenizer.word_index.items() }
rev_spa_char_index = { i: char for char, i in spa_tokenizer.word_index.items() }

# 0은 패딩 토큰이므로, 명시적으로 추가 (만약 토크나이저에 <pad> 토큰이 없다면)
# 보통 Tokenizer는 0을 예약해두지만, texts_to_sequences는 0을 사용하지 않음. pad_sequences가 0을 사용.
# <start>, <end> 토큰 때문에 index가 밀릴 수 있으므로, 안전하게 word_index 기반으로 생성
# <pad>에 대한 처리가 필요하면 명시적으로 추가해야함. 여기서는 0 인덱스는 사용되지 않는다고 가정.

In [366]:
# spa_tokenizer.word_index

In [367]:
# 번역 함수
def decode_seq(input_seq):
    # 입력 시퀀스 인코딩해서 상태 벡터 만들기
    states_value = encoder_model.predict(input_seq, verbose=0) # 다음 예측을위해 직전값 저장

    # 타겟 시퀀스의 시작 토큰(<start>) 생성
    target_seq = np.zeros((1,1)) # (배치크기, 시퀀스길이=1)
    target_seq[0, 0] = spa_tokenizer.word_index['<start>'] # 다음 예측을위해 직전값 저장

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        # 이전 상태, 이전 단어로 다음 단어 예측
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # 예측된 토큰(가장 확률 높은 단어 선택)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        # 인덱스를 단어로
        # sampled_token_index=0(패딩토큰)이면 단어 사전에 없음
        if sampled_token_index == 0: # 패딩 토큰 처리
            sampled_char = ''
        else:
            sampled_char = rev_spa_char_index.get(sampled_token_index, '')

        # 예측된 단어가 <end> 토큰이거나 최대 길이에 도달하면 중단
        if sampled_char == '<end>' or len(decoded_sentence.split()) > max_spa_seq_len:
            stop_condition = True
            break

        if sampled_char: # 빈 문자열이 아니면 단어간 공백 추가
            decoded_sentence += sampled_char + ' '

        # 다음 입력으로 사용할 현재 예측된 단어 업데이트
        target_seq = np.zeros((1,1))
        target_seq[0, 0] = sampled_token_index

        # 상태 업데이트
        states_value = [h, c]

    return decoded_sentence.strip() # ' ' 추가할때 마지막 들어간거 삭제하기

In [368]:
# # 번역 고고
for i in range(10):
    input_seq_test = encoder_input_data[i: i+1] # (1,) -> (1,4)
    org_input_txt = text_pairs[i, 0]
    org_target_txt = text_pairs[i, 1]

    decoded_txt = decode_seq(input_seq_test)

    print(f'{input_seq_test}')
    print(f'입력: {org_input_txt}')
    print(f'정답: {org_target_txt}')
    print(f'번역: {decoded_txt}')
    print('-' * 20)
    
# print(encoder_input_data[i: i+1].shape)
# print(encoder_input_data.shape)
# print(encoder_input_data[i].reshape((1,-1)).shape)

[[ 1 17  2  0  0  0  0]]
입력: <start> go <end>
정답: <start> ve <end>
번역: ve
--------------------
[[ 1 17  2  0  0  0  0]]
입력: <start> go <end>
정답: <start> vete <end>
번역: ve
--------------------
[[ 1 17  2  0  0  0  0]]
입력: <start> go <end>
정답: <start> vaya <end>
번역: ve
--------------------
[[ 1 17  2  0  0  0  0]]
입력: <start> go <end>
정답: <start> váyase <end>
번역: ve
--------------------
[[  1 325   2   0   0   0   0]]
입력: <start> hi <end>
정답: <start> hola <end>
번역: hola
--------------------
[[  1 177   2   0   0   0   0]]
입력: <start> run <end>
정답: <start> ¡corre <end>
번역: corred
--------------------
[[  1 177   2   0   0   0   0]]
입력: <start> run <end>
정답: <start> corred <end>
번역: corred
--------------------
[[ 1 50  2  0  0  0  0]]
입력: <start> who <end>
정답: <start> ¿quién <end>
번역: ¿quién
--------------------
[[  1 370   2   0   0   0   0]]
입력: <start> fire <end>
정답: <start> ¡fuego <end>
번역: ¡incendio
--------------------
[[  1 370   2   0   0   0   0]]
입력: <start> fire <end>
정답: <start

In [369]:
# 직접 문장 입력해서 번역 돌려보기
def translate_user_input(eng_sentence):
    eng_sentence = preprocess_sentence(eng_sentence)
    input_seq = eng_tokenizer.texts_to_sequences([eng_sentence])
    input_seq = pad_sequences(
        input_seq,
        maxlen=max_eng_seq_len,
        padding='post'
    )

    print(f'Preprocessed: {eng_sentence}')
    print(f'Padded: {input_seq}')

    # 없는 단어 처리
    if not np.any(input_seq):
        return '번역불가! 모르는 단어 발견!'

    dec = decode_seq(input_seq)
    print(f'Dec: {dec}')
    return dec

In [371]:
print(translate_user_input('What is your name'))

Preprocessed: <start> what is your name <end>
Padded: [[  1  51   6  49 347   2   0]]
Dec: ¿qué es el arte
¿qué es el arte
